In [38]:
import json
import os
import openai
from pathlib import Path

In [39]:
LEVEL = "B2"
LANGUAGE = "French" 

CHUNK_NUMBER = 1
CHUNK_FOLDER = Path("../../data/chunks")

CURRENT_CHUNK_PATH = CHUNK_FOLDER / f"chunk_{CHUNK_NUMBER}.txt"
PAST_TEXT_PATH = Path("../data/processed/past_text.txt")

# Function to load text from a file
def load_text_file(file_path):
    """Load text content from a file, return None if file doesn't exist or is empty."""
    try:
        if file_path.exists():
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                return content if content else None
        return None
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

# Load the actual text content
CURRENT_CHUNK = load_text_file(CURRENT_CHUNK_PATH)
PAST_TEXT = load_text_file(PAST_TEXT_PATH)

In [40]:
print(CURRENT_CHUNK)

'''3 May. Bistritz._--Left Munich at 8:35 P. M., on 1st May, arriving at
Vienna early next morning; should have arrived at 6:46, but train was an
hour late. Buda-Pesth seems a wonderful place, from the glimpse which I
got of it from the train and the little I could walk through the
streets. I feared to go very far from the station, as we had arrived
late and would start as near the correct time as possible. The
impression I had was that we were leaving the West and entering the
East; the most western of splendid bridges over the Danube, which is
here of noble width and depth, took us among the traditions of Turkish
rule.

We left in pretty good time, and came after nightfall to Klausenburgh.
Here I stopped for the night at the Hotel Royale. I had for dinner, or
rather supper, a chicken done up some way with red pepper, which was
very good but thirsty. (_Mem._, get recipe for Mina.) I asked the
waiter, and he said it was called “paprika hendl,” and that, as it was a
national dish, I sho

In [41]:
print(PAST_TEXT)

None


In [42]:
system_prompt = f"""
You are a patient education assistant fluent in many languages who rewrites book passages so learners can read with comfort and joy.

TARGET LEVEL: {LEVEL} (CEFR)

CORE RULES (apply all):
- Preserve all facts, names, places, dates, and event sequences exactly
- Do NOT add events, characters, or details not in the original
- Maintain a friendly, engaging narrative voice
- Simplify syntax and vocabulary to match {LEVEL} precisely
- Keep original paragraph structure
- Avoid bullet lists unless in the source text
- Explain up to 5 challenging words inline: (word: brief definition) - first occurrence only
- Signal any omissions with [...] - keep minimal
- Preserve proper nouns unchanged
- Maintain the emotional tone and intent of any dialogue or quotes

LEVEL-SPECIFIC GUIDELINES:
- A2: Simple sentences (8-12 words max), basic connectors (and, but, so, then), only high-frequency vocabulary (1000 most common words)
- B1: Mix of simple and compound sentences, everyday vocabulary, clear time/cause connectors (because, when, after, before)
- B2: Varied sentence structures, some abstract concepts allowed, natural flow with sophisticated connectors
- C1: Complex ideas expressed clearly, figurative language preserved when essential, rich but accessible vocabulary
- C2: Near-original sophistication with subtle clarifications, preserve literary style and nuance

OUTPUT: Provide only the simplified text without meta-commentary.
"""

In [43]:
user_prompt = f"""
I have {LEVEL} level {LANGUAGE}. Please simplify this passage into a friendly, story-like narrative.

{f'''CONTEXT FROM PREVIOUS SECTION:
{PAST_TEXT}

Use this only for continuity - do not repeat or rewrite this content.
''' if PAST_TEXT else ''}

TEXT TO SIMPLIFY:
{CURRENT_CHUNK}
"""

In [44]:
print(user_prompt)


I have B2 level French. Please simplify this passage into a friendly, story-like narrative.



TEXT TO SIMPLIFY:
'''3 May. Bistritz._--Left Munich at 8:35 P. M., on 1st May, arriving at
Vienna early next morning; should have arrived at 6:46, but train was an
hour late. Buda-Pesth seems a wonderful place, from the glimpse which I
got of it from the train and the little I could walk through the
streets. I feared to go very far from the station, as we had arrived
late and would start as near the correct time as possible. The
impression I had was that we were leaving the West and entering the
East; the most western of splendid bridges over the Danube, which is
here of noble width and depth, took us among the traditions of Turkish
rule.

We left in pretty good time, and came after nightfall to Klausenburgh.
Here I stopped for the night at the Hotel Royale. I had for dinner, or
rather supper, a chicken done up some way with red pepper, which was
very good but thirsty. (_Mem._, get recipe fo

In [ ]:
client = openai.OpenAI(
    api_key=os.getenv("SWISS_AI_PLATFORM_API_KEY"),
    base_url="https://api.swisscom.com/layer/swiss-ai-weeks/apertus-70b/v1"
)

stream = client.chat.completions.create(
    model="swiss-ai/Apertus-70B",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    stream=True
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

# update chunk number etc

On May 3rd, I left Munich by train at 8:35 PM on May 1st. The train was supposed to arrive in Vienna early the next morning at 6:46, but it was an hour late. 

Vienna is a beautiful city with grand buildings and a wide, deep Danube River. I caught a glimpse of it and walked a bit through the streets. I didn't explore too much because we arrived late and had to leave early. As we left Vienna, it felt like we were moving from the West to the East. The train crossed a grand bridge over the Danube, reminding me of the area's history under Turkish rule.

We arrived in a city called Klausenburg (now called Cluj-Napoca) at dusk. I stayed at the Hotel Royale. 
- For dinner, I had “paprika hendl” (a chicken dish with red pepper), which was tasty but made me very thirsty. 
- The waiter explained that this dish is national, so I could find it in many places along the Carpathian Mountains. 
- I found my limited German useful here; it's handy to know some local language, especially when traveling!
